In [1]:
import pandas as pd
from Bio import SeqIO, Entrez
from Bio.Seq import Seq 
import matplotlib as mpl
from matplotlib.font_manager import FontProperties
mpl.rcParams['svg.fonttype'] = 'none'
# Path to your Arial.ttf file
# arial_font_path = '/home/skawakubo/fonts/Arial.ttf'
# Create a FontProperties object for the Arial font
# arial_font = FontProperties(fname=arial_font_path)
# mpl.rcParams['font.family'] = arial_font.get_name()
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import subprocess

In [2]:
# stem_dir_path = '/Users/skawakubo/github_repos/FCoV-evolution'

In [3]:
def fasta_to_dataframe(fasta_file):
    ids = []
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        ids.append(record.id)
        sequences.append(str(record.seq))

    df = pd.DataFrame({
        "ID": ids,
        "seq": sequences
    })
    return df

In [44]:
# Read the fasta file
AlphaCoV_S_path = 'FCoV_relatives_Spike2_20260219_mafft.fasta'
AlphaCoV_S = fasta_to_dataframe(AlphaCoV_S_path)
AlphaCoV_S.rename(columns={"ID": "Accession"}, inplace=True)
AlphaCoV_S

,Accession,seq
0,NC_002306.3,---------------atgattgtgctcgtaacttgcctcttgttgt...
1,PV941793.1,---------------atgattgtgctcgtaacttgcctcttgttgt...
2,PX471051.1,---------------atgattgtgctcgtaacttgcctcttgttgt...
3,PQ416775.1,---------atgatagtgttaatacttgcacttcttagcaccgcca...
4,PX315748.1,---------atgatagtgctaatacttgcattcgtaagtgtcgtta...
...,...,...
396,MT821905.1,---------atgaagtttttaat-----------------------...
397,MN692770.1,---------atgaaggttttaat-----------------------...
398,LT545990.1,---------atgaagtttttaat-----------------------...
399,KR061459.1,---------atgaagtttttaat-----------------------...


In [45]:
# Read the metadata
FCoV_relatives_meta_path = 'FCoV_relatives_meta_20260218.tsv'
FCoV_relatives_meta = pd.read_csv(FCoV_relatives_meta_path, sep="\t")
FCoV_relatives_meta

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type
0,NC_002306.3,FIPV,Feline infectious peritonitis virus,79-1146,Alphacoronavirus suis,2,29355,USA,USA,NaN,NaN,NaN,2000-05-11,NCBI,FCoV
1,PV941793.1,FIPV,Feline coronavirus,NaN,Alphacoronavirus suis,NaN,29352,USA: California,USA,Felis catus,NaN,2002,2026-01-22,NCBI,FCoV
2,PX471051.1,FIPV,Feline coronavirus,AH2023,Alphacoronavirus suis,NaN,29013,China,China,Feliformia,peritoneal effusion,2023-03-15,2025-10-28,NCBI,FCoV
3,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,NaN,29201,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,FCoV
4,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,1,28497,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,FCoV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399,MT821905.1,NaN,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28081,Italy,Italy,Sus scrofa,NaN,2019-03-04,2022-04-30,NCBI,AlphaCoV1
400,MN692770.1,NaN,Swine enteric coronavirus,SeCoV-1480-Murcia-Lorca,Alphacoronavirus suis,NaN,28094,Spain,Spain,Sus scrofa,NaN,2014-05-30,2020-06-24,NCBI,AlphaCoV1
401,LT545990.1,NaN,Swine enteric coronavirus,SeCoV/GER/L00930/2012,Alphacoronavirus suis,NaN,28050,Germany,Germany,Sus scrofa,NaN,2012,2016-05-14,NCBI,AlphaCoV1
402,KR061459.1,NaN,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28111,Italy,Italy,Sus scrofa,NaN,2009-09-04,2015-12-13,NCBI,AlphaCoV1


In [15]:
# Merge the metadata and the sequence
FCoV_relatives = pd.merge(FCoV_relatives_meta, AlphaCoV_S, on="Accession")
FCoV_relatives

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type,seq
0,NC_002306.3,FIPV,Feline infectious peritonitis virus,79-1146,Alphacoronavirus suis,2,29355,USA,USA,NaN,NaN,NaN,2000-05-11,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
1,PV941793.1,FIPV,Feline coronavirus,NaN,Alphacoronavirus suis,NaN,29352,USA: California,USA,Felis catus,NaN,2002,2026-01-22,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
2,PX471051.1,FIPV,Feline coronavirus,AH2023,Alphacoronavirus suis,NaN,29013,China,China,Feliformia,peritoneal effusion,2023-03-15,2025-10-28,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
3,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,NaN,29201,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,FCoV,---------atgatagtgttaatacttgcacttcttagcaccgcca...
4,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,1,28497,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,FCoV,---------atgatagtgctaatacttgcattcgtaagtgtcgtta...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,MT821905.1,NaN,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28081,Italy,Italy,Sus scrofa,NaN,2019-03-04,2022-04-30,NCBI,AlphaCoV1,---------atgaagtttttaat-----------------------...
396,MN692770.1,NaN,Swine enteric coronavirus,SeCoV-1480-Murcia-Lorca,Alphacoronavirus suis,NaN,28094,Spain,Spain,Sus scrofa,NaN,2014-05-30,2020-06-24,NCBI,AlphaCoV1,---------atgaaggttttaat-----------------------...
397,LT545990.1,NaN,Swine enteric coronavirus,SeCoV/GER/L00930/2012,Alphacoronavirus suis,NaN,28050,Germany,Germany,Sus scrofa,NaN,2012,2016-05-14,NCBI,AlphaCoV1,---------atgaagtttttaat-----------------------...
398,KR061459.1,NaN,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28111,Italy,Italy,Sus scrofa,NaN,2009-09-04,2015-12-13,NCBI,AlphaCoV1,---------atgaagtttttaat-----------------------...


In [16]:
FCoV = FCoV_relatives[FCoV_relatives['virus_type'] == 'FCoV']
FCoV

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type,seq
0,NC_002306.3,FIPV,Feline infectious peritonitis virus,79-1146,Alphacoronavirus suis,2,29355,USA,USA,NaN,NaN,NaN,2000-05-11,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
1,PV941793.1,FIPV,Feline coronavirus,NaN,Alphacoronavirus suis,NaN,29352,USA: California,USA,Felis catus,NaN,2002,2026-01-22,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
2,PX471051.1,FIPV,Feline coronavirus,AH2023,Alphacoronavirus suis,NaN,29013,China,China,Feliformia,peritoneal effusion,2023-03-15,2025-10-28,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
3,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,NaN,29201,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,FCoV,---------atgatagtgttaatacttgcacttcttagcaccgcca...
4,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,1,28497,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,FCoV,---------atgatagtgctaatacttgcattcgtaagtgtcgtta...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,PQ133191.1,FIPV,Alphacoronavirus 1,2-G2 Ka 6882,Alphacoronavirus suis,2,28381,Cyprus,Cyprus,Felis catus,Effusion fluid,NaN,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
391,PQ133192.1,FIPV,Alphacoronavirus 1,2-H6 Ka 7790,Alphacoronavirus suis,2,28517,Cyprus,Cyprus,Felis catus,Effusion fluid,2023-08-06,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
392,PQ133193.1,FIPV,Alphacoronavirus 1,2-H7 Ja 8138,Alphacoronavirus suis,2,28487,Cyprus,Cyprus,Felis catus,cerebrospinal fluid,2023-06-16,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
393,PQ133194.1,FIPV,Alphacoronavirus 1,2-H8 So 8842,Alphacoronavirus suis,2,28470,Cyprus,Cyprus,Felis catus,Effusion fluid,2023-06-07,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...


In [17]:
FCoV['biotype'].value_counts()

biotype
FIPV       95
unknown    85
FECV       38
Name: count, dtype: int64

In [18]:
FCoV_meta = FCoV.drop(columns='seq')
FCoV_meta

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type
0,NC_002306.3,FIPV,Feline infectious peritonitis virus,79-1146,Alphacoronavirus suis,2,29355,USA,USA,NaN,NaN,NaN,2000-05-11,NCBI,FCoV
1,PV941793.1,FIPV,Feline coronavirus,NaN,Alphacoronavirus suis,NaN,29352,USA: California,USA,Felis catus,NaN,2002,2026-01-22,NCBI,FCoV
2,PX471051.1,FIPV,Feline coronavirus,AH2023,Alphacoronavirus suis,NaN,29013,China,China,Feliformia,peritoneal effusion,2023-03-15,2025-10-28,NCBI,FCoV
3,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,NaN,29201,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,FCoV
4,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,1,28497,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,FCoV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,PQ133191.1,FIPV,Alphacoronavirus 1,2-G2 Ka 6882,Alphacoronavirus suis,2,28381,Cyprus,Cyprus,Felis catus,Effusion fluid,NaN,2025-02-01,NCBI,FCoV
391,PQ133192.1,FIPV,Alphacoronavirus 1,2-H6 Ka 7790,Alphacoronavirus suis,2,28517,Cyprus,Cyprus,Felis catus,Effusion fluid,2023-08-06,2025-02-01,NCBI,FCoV
392,PQ133193.1,FIPV,Alphacoronavirus 1,2-H7 Ja 8138,Alphacoronavirus suis,2,28487,Cyprus,Cyprus,Felis catus,cerebrospinal fluid,2023-06-16,2025-02-01,NCBI,FCoV
393,PQ133194.1,FIPV,Alphacoronavirus 1,2-H8 So 8842,Alphacoronavirus suis,2,28470,Cyprus,Cyprus,Felis catus,Effusion fluid,2023-06-07,2025-02-01,NCBI,FCoV


In [ ]:
FCoV_meta.to_csv('FCoV_meta20260219.tsv', sep='\t', index=False)

In [22]:
# write out as fasta file
with open('FCoV_S_20260219.fasta', 'w') as f:
    for index, row in FCoV.iterrows():
        f.write(f">{row['Accession']}\n{row['seq']}\n")

In [56]:
list_of_type1 = ['MT821905.1','LT545990.1','MN692770.1','NC_028806.1','KR061459.1','KF530116.1','KP849472.1','c57','KF530125.1','KF530129.1','MH817484.1','JN183883.1','KF530134.1','KF530136.1','KF530127.1','KF530130.1','KF530131.1','KF530120.1','KF530126.1','KF530123.1','KF530135.1','JN183882.1','PX315755.1','PX315756.1','HQ012370.1','PX315757.1','PX315758.1','c66','PX315749.1','PX315750.1','KF530119.1','KF530122.1','KF530124.1','KF530270.1','KF530271.1','ON595866.1','ON595868.1','ON595863.1','ON595853.1','ON595867.1','ON595869.1','ON595871.1','ON595857.1','ON595862.1','ON595870.1','ON595854.1','ON595855.1','HQ392470.1','HQ012367.1','ON595856.1','DQ848678.1','PX315754.1','PX315751.1','PX315752.1','PX315753.1','KP143511.1','KP143510.1','KP143509.1','KP143508.1','KP143507.1','KP143512.1','c34','771819','c77','772363','22N2321','ON595858.1','v7','c46','KF530117.1','MG893511.1','FJ938058.1','HQ012372.1','HQ012369.1','GU553361.1','GU553362.1','HQ012368.1','FJ938057.1','FJ938056.1','FJ938053.1','FJ938054.1','FJ938051.1','KF530132.1','HQ392471.1','EU186072.1','FJ938060.1','KU215423.1','KU215426.1','KU215421.1','KU215422.1','KU215427.1','KU215428.1','FJ938061.1','KU215419.1','KU215420.1','KU215424.1','KU215425.1','KX722531.1','c33','FJ938062.1','FJ938055.1','FJ938052.1','FJ938059.1','PQ416775.1','PP908790.1','MT239440.1','KX722529.1','775024','PP908788.1','MT444152.1','KY292377.1','KF530118.1','PX315748.1','PP254210.1','PP254211.1','PP254209.1','PP254213.1','PP254215.1','PP254208.1','PP254212.1','PP254206.1','PP254214.1','PV780394.1','PP254207.1','MW030110.1','PV797401.1','KF530121.1','KF530133.1','MW308128.1','PV797400.1','PV797403.1','PP908789.1','MW030108.1','PV780395.1','KY566211.1','KY566209.1','KY566210.1','c112','HQ392469.1','HQ392472.1','HQ012371.1','OP542206.1','MN165107.1','c63','PP854701.1','PP854702.1','v11','r2','LC742526.1','v2','v1','LC852817.1','v5','h7','h2','LC880185.1','r1','KX722530.1','v4','fecv15','fecv30','fecv14','fecv22','fecv5','fecv_NARC3','fecv_NARC4','fecv23','fecv43','fecv40','h6','fecv60','fecv50','fecv47','fecv16','fecv46','r3','fecv28','fecv37','fecv51','fecv66']
print(len(list_of_type1))

180


In [57]:
FCoV_relatives_type1 = FCoV_relatives[FCoV_relatives['Accession'].isin(list_of_type1)]
FCoV_relatives_type1.reset_index(inplace=True)
FCoV_relatives_type1

,index,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type,seq
0,3,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,NaN,29201,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,FCoV,---------atgatagtgttaatacttgcacttcttagcaccgcca...
1,4,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,1,28497,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,FCoV,---------atgatagtgctaatacttgcattcgtaagtgtcgtta...
2,5,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,1,28550,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI,FCoV,---------atgatagtgctaatacttgcattcgtaagtgttgtta...
3,6,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,1,29246,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI,FCoV,---------atgatagtgctaatacttgcattcgtaagtgttgtta...
4,7,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,1,29225,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI,FCoV,---------atgatagtgttaatacttgcactccttagctttgcta...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,395,MT821905.1,NaN,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28081,Italy,Italy,Sus scrofa,NaN,2019-03-04,2022-04-30,NCBI,AlphaCoV1,---------atgaagtttttaat-----------------------...
176,396,MN692770.1,NaN,Swine enteric coronavirus,SeCoV-1480-Murcia-Lorca,Alphacoronavirus suis,NaN,28094,Spain,Spain,Sus scrofa,NaN,2014-05-30,2020-06-24,NCBI,AlphaCoV1,---------atgaaggttttaat-----------------------...
177,397,LT545990.1,NaN,Swine enteric coronavirus,SeCoV/GER/L00930/2012,Alphacoronavirus suis,NaN,28050,Germany,Germany,Sus scrofa,NaN,2012,2016-05-14,NCBI,AlphaCoV1,---------atgaagtttttaat-----------------------...
178,398,KR061459.1,NaN,Swine enteric coronavirus,NaN,Alphacoronavirus suis,NaN,28111,Italy,Italy,Sus scrofa,NaN,2009-09-04,2015-12-13,NCBI,AlphaCoV1,---------atgaagtttttaat-----------------------...


In [49]:
list_of_type2 = ['OP179857.1','MT906864.1','JN856008.2','OX335534.1','OX335535.1','OX335546.1','OX335530.1','OX335545.1','OX335548.1','OQ540912.1','GQ477367.1','OX335549.1','MT906865.1','PQ383370.1','OM950728.1','OM950729.1','PP526172.1','MZ420153.1','MW591993.2','OM830318.1','OM830320.1','OK078898.1','OR689864.1','OR689863.1','OR209253.1','OM830321.1','DQ811787.1','OM830319.1','PV096984.1','OR209251.1','OR209252.1','KY406735.1','OR209254.1','KR270796.1','KX900395.1','KX900396.1','KX900397.1','KX900400.1','KX900401.1','KX900403.1','KX900398.1','KX900409.1','KX900408.1','KX900410.1','KX900399.1','KX900404.1','KX900406.1','KX900405.1','KX900407.1','KX900411.1','PX137150.1','DQ811786.2','MT522161.1','FJ755618.2','OP805351.1','EU074218.2','MZ322950.1','MW804449.1','PQ189446.1','ON858825.1','OP434397.1','PQ856105.1','PQ821888.1','KX058075.1','DQ201447.1','OZ259977.1','DQ811785.1','KT696544.1','KX499468.1','DQ811789.2','ON016092.1','DQ443743.1','OR085313.1','KX900393.1','KX900402.1','KX900394.1','ON859974.1','OM802899.1','PX756779.1','PX756778.1','PP236916.1','MZ368889.1','KP202848.1','HM776941.1','DQ811788.1','KC962433.1','KX083668.1','KU729220.1','NC_038861.1','MT576083.1','HQ462571.1','CQ870486.1','AX154950.1','AJ271965.2','PP526171.1','JN634064.1','JQ404410.1','KY063616.1','KY063617.1','KY063618.2','JQ408980.1','JQ408981.1','PX471051.1','OR295209.1','OR711916.1','DQ286389.1','PQ192636.1','PV941793.1','PQ202302.1','OQ311323.1','KC461235.1','MW030109.1','DQ010921.1','KC461236.1','NC_002306.3','PP526173.1','KC461237.1','AY994055.1','KC175339.1','PQ383369.1','GQ152141.1','ON107244.1','OR991746.1','PV797402.1','MT239439.1','MT114538.1','MT114541.1','MT114543.1','MT114539.1','MT114544.1','MT114540.1','MT114542.1','PP526170.1','JQ404409.1','KC175340.1','KC175341.1','OM451123.1','OM864519.1','OM864518.1','OM451122.1','OQ540913.1','OQ540914.1','OQ540915.1','OQ540909.1','OQ540918.1','OQ540917.1','OQ540908.1','MW354910.1','OQ540910.1','OQ540916.1','OQ540921.1','OQ540911.1','OQ540919.1','OQ540922.1','OQ540920.1','MW354911.1','MZ320953.1','MZ320954.1','OX335541.1','OX335623.1','PQ177897.1','PQ127013.1','PQ127011.1','PQ127012.1','KP981644.1','PQ177896.1','c47','PQ133187.1','PQ133186.1','PQ133176.1','PQ133189.1','PQ133185.1','PQ133191.1','PQ133180.1','PQ133188.1','PQ133195.1','PQ133181.1','PQ133179.1','PQ133183.1','PQ133177.1','PQ133192.1','PQ133182.1','PQ133194.1','PQ133178.1','PQ133184.1','PQ133193.1','PQ133190.1']
print(len(list_of_type2))

187


In [54]:
# write out as fasta file
with open('FCoV_relatives_type1.fasta', 'w') as f:
    for index, row in FCoV_relatives_type1.iterrows():
        f.write(f">{row['Accession']}\n{row['seq']}\n")

In [50]:
FCoV_relatives_type2 = FCoV_relatives[FCoV_relatives['Accession'].isin(list_of_type2)]
FCoV_relatives_type2

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type,seq
0,NC_002306.3,FIPV,Feline infectious peritonitis virus,79-1146,Alphacoronavirus suis,2,29355,USA,USA,NaN,NaN,NaN,2000-05-11,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
1,PV941793.1,FIPV,Feline coronavirus,NaN,Alphacoronavirus suis,NaN,29352,USA: California,USA,Felis catus,NaN,2002,2026-01-22,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
2,PX471051.1,FIPV,Feline coronavirus,AH2023,Alphacoronavirus suis,NaN,29013,China,China,Feliformia,peritoneal effusion,2023-03-15,2025-10-28,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
19,PV797402.1,FECV,Feline coronavirus,FCoV/Thailand/CU32825/2023,Alphacoronavirus suis,2,28757,Thailand,Thailand,Felis catus,feces,2023-06,2025-06-30,NCBI,FCoV,---------------atgattg---ttctaacattgttcattcttt...
23,PQ192636.1,FIPV,Feline infectious peritonitis virus,FIPV/A15/LIU/2024,Alphacoronavirus suis,NaN,29355,USA,USA,Felis catus,colon tissue,2024,2025-03-26,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,PQ133191.1,FIPV,Alphacoronavirus 1,2-G2 Ka 6882,Alphacoronavirus suis,2,28381,Cyprus,Cyprus,Felis catus,Effusion fluid,NaN,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
391,PQ133192.1,FIPV,Alphacoronavirus 1,2-H6 Ka 7790,Alphacoronavirus suis,2,28517,Cyprus,Cyprus,Felis catus,Effusion fluid,2023-08-06,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
392,PQ133193.1,FIPV,Alphacoronavirus 1,2-H7 Ja 8138,Alphacoronavirus suis,2,28487,Cyprus,Cyprus,Felis catus,cerebrospinal fluid,2023-06-16,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
393,PQ133194.1,FIPV,Alphacoronavirus 1,2-H8 So 8842,Alphacoronavirus suis,2,28470,Cyprus,Cyprus,Felis catus,Effusion fluid,2023-06-07,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...


In [55]:
# write out as fasta file
with open('FCoV_relatives_type2.fasta', 'w') as f:
    for index, row in FCoV_relatives_type2.iterrows():
        f.write(f">{row['Accession']}\n{row['seq']}\n")

In [65]:
list_of_FCoV2_CCoV = ['MZ320953.1','MZ320954.1','PQ177896.1','OX335541.1','OX335623.1','PQ177897.1','PQ127013.1','PQ127011.1','PQ127012.1','KP981644.1','c47','PQ133187.1','PQ133195.1','PQ133191.1','PQ133176.1','PQ133179.1','PQ133177.1','PQ133183.1','PQ133180.1','PQ133189.1','PQ133181.1','PQ133185.1','PQ133186.1','PQ133188.1','PQ133182.1','PQ133194.1','PQ133192.1','PQ133178.1','PQ133184.1','PQ133190.1','PQ133193.1','PQ383369.1','GQ152141.1','ON107244.1','OR991746.1','PV797402.1','MT239439.1','MT114538.1','MT114541.1','MT114543.1','MT114539.1','MT114544.1','MT114540.1','MT114542.1','PP526171.1','JN634064.1','JQ404410.1','KY063616.1','KY063617.1','JQ408981.1','PX471051.1','OR711916.1','OR295209.1','DQ286389.1','MW030109.1','OQ311323.1','KC461235.1','PQ192636.1','PV941793.1','PQ202302.1','DQ010921.1','KC461236.1','NC_002306.3','PP526173.1']
print(len(list_of_FCoV2))

64


In [66]:
FCoV2 = FCoV_relatives[FCoV_relatives['Accession'].isin(list_of_FCoV2_CCoV)]
FCoV2

,Accession,biotype,Organism_Name,Isolate,Species,Genotype,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,virus_type,seq
0,NC_002306.3,FIPV,Feline infectious peritonitis virus,79-1146,Alphacoronavirus suis,2,29355,USA,USA,NaN,NaN,NaN,2000-05-11,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
1,PV941793.1,FIPV,Feline coronavirus,NaN,Alphacoronavirus suis,NaN,29352,USA: California,USA,Felis catus,NaN,2002,2026-01-22,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
2,PX471051.1,FIPV,Feline coronavirus,AH2023,Alphacoronavirus suis,NaN,29013,China,China,Feliformia,peritoneal effusion,2023-03-15,2025-10-28,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
19,PV797402.1,FECV,Feline coronavirus,FCoV/Thailand/CU32825/2023,Alphacoronavirus suis,2,28757,Thailand,Thailand,Felis catus,feces,2023-06,2025-06-30,NCBI,FCoV,---------------atgattg---ttctaacattgttcattcttt...
23,PQ192636.1,FIPV,Feline infectious peritonitis virus,FIPV/A15/LIU/2024,Alphacoronavirus suis,NaN,29355,USA,USA,Felis catus,colon tissue,2024,2025-03-26,NCBI,FCoV,---------------atgattgtgctcgtaacttgcctcttgttgt...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,PQ133191.1,FIPV,Alphacoronavirus 1,2-G2 Ka 6882,Alphacoronavirus suis,2,28381,Cyprus,Cyprus,Felis catus,Effusion fluid,NaN,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
391,PQ133192.1,FIPV,Alphacoronavirus 1,2-H6 Ka 7790,Alphacoronavirus suis,2,28517,Cyprus,Cyprus,Felis catus,Effusion fluid,2023-08-06,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
392,PQ133193.1,FIPV,Alphacoronavirus 1,2-H7 Ja 8138,Alphacoronavirus suis,2,28487,Cyprus,Cyprus,Felis catus,cerebrospinal fluid,2023-06-16,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...
393,PQ133194.1,FIPV,Alphacoronavirus 1,2-H8 So 8842,Alphacoronavirus suis,2,28470,Cyprus,Cyprus,Felis catus,Effusion fluid,2023-06-07,2025-02-01,NCBI,FCoV,---------------atgattgtgatcgtaacttgcatcctatcat...


In [67]:
# write out as fasta file
with open('FCoV2_CCoV_trim.fasta', 'w') as f:
    for index, row in FCoV2.iterrows():
        f.write(f">{row['Accession']}\n{row['seq']}\n")